# Cross-resonance calibration workflow

This notebook sketches a practical CR bring-up for one coupled qubit pair:
prepare the single-qubit pulse set, obtain CR parameters, calibrate a
`ZX90` gate, inspect the resulting schedule, and validate it with Bell
measurements and interleaved RB.


## 1. Create an `Experiment`

Pick a known coupled pair for your device. The example below uses the
first two qubits from the selected subset, but you should replace them
with a pair that has a calibrated CR path in your system.


In [ ]:
import numpy as np

import qubex as qx

exp = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    qubits=[0, 1],
    configuration_mode="ge-ef-cr",
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

if len(exp.qubit_labels) < 2:
    raise ValueError("Select at least two coupled qubits for CR calibration.")

cr_control, cr_target = exp.qubit_labels[:2]
cr_label = f"{cr_control}-{cr_target}"

print("control qubit:", cr_control)
print("target qubit:", cr_target)
print("pair label:", cr_label)

## 2. Connect and prepare the single-qubit prerequisites

CR calibration depends on decent single-qubit control and a usable readout
classifier on both qubits.


In [ ]:
exp.connect()

rabi_result = exp.obtain_rabi_params(
    targets=[cr_control, cr_target],
    n_shots=512,
    plot=True,
)
hpi_result = exp.calibrate_hpi_pulse(
    targets=[cr_control, cr_target],
    n_shots=512,
    plot=True,
)
pi_result = exp.calibrate_pi_pulse(
    targets=[cr_control, cr_target],
    n_shots=512,
    plot=True,
)
drag_hpi_result = exp.calibrate_drag_hpi_pulse(
    targets=[cr_control, cr_target],
    n_iterations=2,
    n_shots=512,
    plot=True,
)
drag_pi_result = exp.calibrate_drag_pi_pulse(
    targets=[cr_control, cr_target],
    n_iterations=2,
    n_shots=512,
    plot=True,
)
classifier_result = exp.build_classifier(
    targets=[cr_control, cr_target],
    n_states=2,
    save_classifier=False,
    n_shots=4096,
    plot=True,
)

## 3. Obtain the CR interaction parameters


In [ ]:
cr_params_result = exp.obtain_cr_params(
    cr_control,
    cr_target,
    time_range=np.arange(64, 641, 32),
    n_iterations=2,
    n_shots=512,
    plot=True,
)

## 4. Calibrate a `ZX90` gate


In [ ]:
zx90_calibration_result = exp.calibrate_zx90(
    control_qubit=cr_control,
    target_qubit=cr_target,
    n_repetitions=5,
    n_shots=512,
    plot=True,
)

## 5. Inspect the cached `ZX90` schedule and repeat it


In [ ]:
zx90 = exp.zx90(cr_control, cr_target)
zx90.plot()

repeat_result = exp.repeat_sequence(zx90)

## 6. Run pulse tomography from two control-qubit initial states


In [ ]:
pulse_tomography_0 = exp.pulse_tomography(
    zx90.repeated(4),
    initial_state={cr_control: "0"},
    n_shots=1024,
    plot=True,
)

pulse_tomography_1 = exp.pulse_tomography(
    zx90.repeated(4),
    initial_state={cr_control: "1"},
    n_shots=1024,
    plot=True,
)

## 7. Validate the entangling gate with Bell-state measurement and IRB


In [ ]:
bell_result = exp.measure_bell_state(
    cr_control,
    cr_target,
    zx90=zx90,
    n_shots=1024,
    plot=True,
    save_image=False,
)

irb_result = exp.interleaved_randomized_benchmarking(
    cr_label,
    interleaved_clifford="ZX90",
    n_cliffords_range=np.arange(0, 101, 10),
    n_trials=20,
    n_shots=512,
    plot=True,
    save_image=False,
)

## 8. Optionally persist the CR calibration


In [ ]:
# exp.calib_note.save()
